# Tomato Run 07: CutMix
Run 05 with CutMix, alpha 1.0 (single change).

## 1. GPU check

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 2. Experiment settings + Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q imagehash
from pathlib import Path
import json, time, gc, hashlib
import numpy as np, pandas as pd, psutil, imagehash
from PIL import Image, ImageOps
from tqdm.auto import tqdm

PROJECT  = Path('/content/drive/MyDrive/thesis')
ZIP_PATH = PROJECT / 'data/raw/agri_vision.zip'
# ===================== EXPERIMENT SETTINGS =====================
RUN_NAME        = 'run07_cutmix'           # a new name for every run
LABEL_SMOOTHING = 0.0                      # Run 03: 0.1
TARGET_PER_CLASS = 300                     # Run 04: e.g. 300 = every class topped up to 300 augmented training images; None = original training set
USE_CLASS_WEIGHTS = False                  # set False when TARGET_PER_CLASS balances the classes
AUGMENT         = 'moderate'               # Run 05: 'basic' = Runs 01b-04 augmentation, 'moderate' = stronger, more varied
MIXUP_ALPHA     = 0                        # Run 06 used 0.2 (rejected); 0 = off
CUTMIX_ALPHA    = 1.0                      # Run 07: CutMix strength; 0 = off (Runs 01b-06)
IMG_SIZE        = 224
MODEL_NAMES     = ['VGG16', 'ConvNeXtTiny', 'EfficientNetV2B0', 'DenseNet121', 'InceptionV3', 'ResNet50']   # top 5 of Run 01 + ResNet50
NOTE            = 'Run 07: Run 05 setup (balanced 300, no class weights, moderate augmentation) + CutMix alpha 1.0; one change vs Run 05'
# ===============================================================
OUT      = PROJECT / 'experiments' / 'tomato' / RUN_NAME
for sub in ['history', 'preds']:
    (OUT / sub).mkdir(parents=True, exist_ok=True)
RESULTS_CSV = OUT / 'results.csv'

CROP, FOLD, SEED, N_FOLDS = 'Tomato', 0, 42, 5
print('Results go to', OUT)

Mounted at /content/drive
Results go to /content/drive/MyDrive/thesis/experiments/tomato/run07_cutmix


## 3. Unzip the images to local disk

In [3]:
UNZIP_DIR = Path('/content/data/agri_vision')
if not UNZIP_DIR.exists():
    UNZIP_DIR.mkdir(parents=True)
    !unzip -q "{ZIP_PATH}" -d "{UNZIP_DIR}"
    for inner in list(UNZIP_DIR.rglob('*.zip')):
        !unzip -q -o "{inner}" -d "{inner.parent}"
ORIG_DIR = [p for p in UNZIP_DIR.rglob('*') if p.is_dir() and 'original' in p.name.lower()][0]
print('Original images folder:', ORIG_DIR)

Original images folder: /content/data/agri_vision/Agri-Vision4 A Comprehensive Multi-Crop Leaf Disea/Bottle Gourd, Papaya, Tomato, and Zucchini Original Dataset


## 4. Audit and clean

In [4]:
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
rows = []
for p in tqdm(sorted(q for q in ORIG_DIR.rglob('*') if q.suffix.lower() in IMG_EXT), desc='Hashing'):
    parts = p.relative_to(ORIG_DIR).parts
    rec = {'rel_path': '/'.join(parts), 'split': parts[0], 'label_raw': parts[-2]}
    try:
        with Image.open(p) as im:
            im.verify()
        with Image.open(p) as im:
            rec['phash'] = str(imagehash.phash(im.convert('RGB')))
        rec['md5'] = hashlib.md5(p.read_bytes()).hexdigest()
        rec['corrupt'] = False
    except Exception:
        rec['corrupt'] = True
    rows.append(rec)
df = pd.DataFrame(rows).sort_values('rel_path').reset_index(drop=True)
df['label'] = df.label_raw.replace({'Tomato_Downy Mildew': 'Tomato_Downy_Mildew', 'Zucchini_Downy_Midew': 'Zucchini_Downy_Mildew'})
df['crop'] = df.label.str.extract(r'^(Bottle_Gourd|Papaya|Tomato|Zucchini)', expand=False)
df['keep'], df['reason'] = True, ''

def drop(mask, reason):
    m = mask & df.keep
    df.loc[m, 'keep'], df.loc[m, 'reason'] = False, reason
    print(f'{reason:24s} removed {int(m.sum()):5d}   remaining {int(df.keep.sum()):5d}')

drop(df.corrupt.astype(bool), 'unreadable')
g = df[df.keep].groupby('md5').label.nunique();   drop(df.md5.isin(g[g > 1].index), 'label_conflict_exact')
drop(df.keep & df.duplicated('md5'), 'duplicate_copy')
g = df[df.keep].groupby('phash').label.nunique(); drop(df.phash.isin(g[g > 1].index), 'label_conflict_near')
drop(df.crop.eq('Zucchini'), 'excluded_zucchini')

Hashing:   0%|          | 0/5246 [00:00<?, ?it/s]

unreadable               removed     0   remaining  5246
label_conflict_exact     removed   754   remaining  4492
duplicate_copy           removed   463   remaining  4029
label_conflict_near      removed    18   remaining  4011
excluded_zucchini        removed   262   remaining  3749


## 5. Five stratified folds, grouped by near-duplicates; select Tomato

In [5]:
from sklearn.model_selection import StratifiedGroupKFold
kept = df[df.keep].copy()
kept['fold'] = -1
for k, (_, te) in enumerate(StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
                            .split(kept, kept.label, groups=kept.phash)):
    kept.iloc[te, kept.columns.get_loc('fold')] = k
assert kept.md5.is_unique and kept.groupby('phash').fold.nunique().max() == 1

d = kept[kept.crop == CROP].sort_values('rel_path').reset_index(drop=True)
CLASSES = sorted(d.label.unique())
y = d.label.map({c: i for i, c in enumerate(CLASSES)}).to_numpy()
te, va = (d.fold == FOLD).to_numpy(), (d.fold == (FOLD + 1) % N_FOLDS).to_numpy()
tr = ~(te | va)
d.to_csv(OUT / 'tomato_split.csv', index=False)
print(f'{CROP}: {len(d)} clean images, {len(CLASSES)} classes')
display(pd.crosstab(d.label, np.select([te, va], ['test', 'validation'], 'train')))

Tomato: 795 clean images, 5 classes


col_0,test,train,validation
label,,,
Tomato_Downy_Mildew,8,36,11
Tomato_Healthy,49,151,46
Tomato_Mosaic,37,113,41
Tomato_Spot,43,152,43
Tomato_White_Spot,11,41,13


## 6. Training settings and the training script

In [6]:
cfg = {'run': RUN_NAME, 'note': NOTE, 'crop': CROP, 'fold': FOLD, 'seed': SEED, 'n_folds': N_FOLDS,
       'img_size': IMG_SIZE, 'batch': 32, 'label_smoothing': LABEL_SMOOTHING, 'target_per_class': TARGET_PER_CLASS, 'class_weights': USE_CLASS_WEIGHTS, 'augment': AUGMENT, 'mixup_alpha': MIXUP_ALPHA, 'cutmix_alpha': CUTMIX_ALPHA, 'models': MODEL_NAMES, 'head_epochs': 15, 'head_lr': 1e-3,
       'ft_epochs': 40, 'ft_lr': 3e-5, 'warmup_epochs': 2, 'ft_patience': 10,
       'optimizer': 'AdamW(weight_decay=1e-4, clipnorm=1.0)',
       'early_stopping': 'head: val_loss p3; fine-tune: val_accuracy p10; restore best',
       'augmentation': 'flip h+v, rotation 0.1, zoom 0.1, contrast 0.1 (train only)',
       'orig_dir': str(ORIG_DIR), 'out': str(OUT)}
(OUT / 'config.json').write_text(json.dumps(cfg, indent=2))
print(json.dumps(cfg, indent=2))

{
  "run": "run07_cutmix",
  "note": "Run 07: Run 05 setup (balanced 300, no class weights, moderate augmentation) + CutMix alpha 1.0; one change vs Run 05",
  "crop": "Tomato",
  "fold": 0,
  "seed": 42,
  "n_folds": 5,
  "img_size": 224,
  "batch": 32,
  "label_smoothing": 0.0,
  "target_per_class": 300,
  "class_weights": false,
  "augment": "moderate",
  "mixup_alpha": 0,
  "cutmix_alpha": 1.0,
  "models": [
    "VGG16",
    "ConvNeXtTiny",
    "EfficientNetV2B0",
    "DenseNet121",
    "InceptionV3",
    "ResNet50"
  ],
  "head_epochs": 15,
  "head_lr": 0.001,
  "ft_epochs": 40,
  "ft_lr": 3e-05,
  "warmup_epochs": 2,
  "ft_patience": 10,
  "optimizer": "AdamW(weight_decay=1e-4, clipnorm=1.0)",
  "early_stopping": "head: val_loss p3; fine-tune: val_accuracy p10; restore best",
  "augmentation": "flip h+v, rotation 0.1, zoom 0.1, contrast 0.1 (train only)",
  "orig_dir": "/content/data/agri_vision/Agri-Vision4 A Comprehensive Multi-Crop Leaf Disea/Bottle Gourd, Papaya, Tomato, and 

In [7]:
%%writefile /content/train_one.py
# Trains ONE model on the split saved by the notebook, then appends its results. Usage: python train_one.py MODEL
import os, sys, json, time
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image, ImageOps
import tensorflow as tf, keras
from keras import layers
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score,
                             cohen_kappa_score, matthews_corrcoef, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight

name = sys.argv[1]
OUT = Path(sys.argv[2]); cfg = json.loads((OUT / 'config.json').read_text())
ORIG_DIR, IMG, BATCH, SEED, FOLD = Path(cfg['orig_dir']), cfg['img_size'], cfg['batch'], cfg['seed'], cfg['fold']
d = pd.read_csv(OUT / 'tomato_split.csv')
CLASSES = sorted(d.label.unique())
y = d.label.map({c: i for i, c in enumerate(CLASSES)}).to_numpy()
te = (d.fold == FOLD).to_numpy(); va = (d.fold == (FOLD + 1) % cfg['n_folds']).to_numpy(); tr = ~(te | va)

A, identity = keras.applications, (lambda x: x)
MODELS = {'VGG16': (A.VGG16, A.vgg16.preprocess_input), 'ResNet50': (A.ResNet50, A.resnet50.preprocess_input),
          'DenseNet121': (A.DenseNet121, A.densenet.preprocess_input),
          'InceptionV3': (A.InceptionV3, A.inception_v3.preprocess_input),
          'Xception': (A.Xception, A.xception.preprocess_input),
          'MobileNetV3Large': (A.MobileNetV3Large, identity),
          'EfficientNetV2B0': (A.EfficientNetV2B0, identity), 'ConvNeXtTiny': (A.ConvNeXtTiny, identity)}
keras.mixed_precision.set_global_policy('float32' if name in {'MobileNetV3Large', 'Xception'} else 'mixed_float16')
keras.utils.set_random_seed(SEED + FOLD)

X = np.stack([np.asarray(ImageOps.pad(Image.open(ORIG_DIR / p).convert('RGB'), (IMG, IMG), color=(0, 0, 0)),
                         dtype=np.uint8) for p in d.rel_path])
# Augmentation (training images only). 'basic' = Runs 01b-04; 'moderate' = Run 05.
# No hue/saturation changes: leaf colour is a disease cue (e.g. yellowing in Mosaic).
if cfg.get('augment', 'basic') == 'moderate':
    augment = keras.Sequential([
        layers.RandomFlip('horizontal_and_vertical', dtype='float32'),
        layers.RandomRotation(0.2, dtype='float32'),             # up to +-72 degrees
        layers.RandomZoom(0.2, dtype='float32'),                 # zoom in/out up to 20 %
        layers.RandomTranslation(0.1, 0.1, dtype='float32'),     # shift up to 10 % sideways / up-down
        layers.RandomContrast(0.2, dtype='float32'),
        layers.RandomBrightness(0.15, value_range=(0, 255), dtype='float32')])
else:
    augment = keras.Sequential([layers.RandomFlip('horizontal_and_vertical', dtype='float32'),
                                layers.RandomRotation(0.1, dtype='float32'), layers.RandomZoom(0.1, dtype='float32'),
                                layers.RandomContrast(0.1, dtype='float32')])
ctor, prep = MODELS[name]

# MixUp (training batches only): blend each photo with another random photo from the same batch.
# lam ~ Beta(alpha, alpha) per image; with alpha 0.2 most blends are mostly one photo (lam near 0 or 1).
# The labels are blended with the same weights, e.g. 0.8 Spot + 0.2 Mosaic.
ALPHA = float(cfg.get('mixup_alpha', 0) or 0)
def mixup(a, b):
    n = tf.shape(a)[0]
    g1 = tf.random.gamma([n], ALPHA); g2 = tf.random.gamma([n], ALPHA)
    lam = g1 / (g1 + g2)                                   # Beta(alpha, alpha) sample per image
    j = tf.random.shuffle(tf.range(n))                     # partner image for each image
    lx, ly = tf.reshape(lam, [-1, 1, 1, 1]), tf.reshape(lam, [-1, 1])
    return a * lx + tf.gather(a, j) * (1 - lx), b * ly + tf.gather(b, j) * (1 - ly)

# CutMix (training batches only): cut a random rectangle out of each photo and paste in the same area
# from another photo of the batch. The label is shared by area, e.g. 70 % Spot + 30 % Mosaic.
CUTMIX_ALPHA = float(cfg.get('cutmix_alpha', 0) or 0)
def cutmix(a, b):
    n, H, W = tf.shape(a)[0], tf.shape(a)[1], tf.shape(a)[2]
    g1 = tf.random.gamma([n], CUTMIX_ALPHA); g2 = tf.random.gamma([n], CUTMIX_ALPHA)
    lam = g1 / (g1 + g2)                                   # share of the own photo that is kept
    r = tf.sqrt(1.0 - lam)
    ch = tf.cast(r * tf.cast(H, tf.float32), tf.int32); cw = tf.cast(r * tf.cast(W, tf.float32), tf.int32)
    cy = tf.random.uniform([n], 0, H, dtype=tf.int32); cx = tf.random.uniform([n], 0, W, dtype=tf.int32)
    y1 = tf.clip_by_value(cy - ch // 2, 0, H); y2 = tf.clip_by_value(cy + ch // 2, 0, H)
    x1 = tf.clip_by_value(cx - cw // 2, 0, W); x2 = tf.clip_by_value(cx + cw // 2, 0, W)
    rows, cols = tf.range(H)[None, :], tf.range(W)[None, :]
    in_y = (rows >= y1[:, None]) & (rows < y2[:, None])
    in_x = (cols >= x1[:, None]) & (cols < x2[:, None])
    m = tf.cast(in_y[:, :, None] & in_x[:, None, :], tf.float32)[..., None]
    j = tf.random.shuffle(tf.range(n))
    area = tf.reduce_mean(m, axis=[1, 2, 3])
    keep = tf.reshape(1.0 - area, [-1, 1])
    return a * (1 - m) + tf.gather(a, j) * m, b * keep + tf.gather(b, j) * (1 - keep)

def make_ds(idx, train):
    ds = tf.data.Dataset.from_tensor_slices((X[idx], y[idx]))
    if train:
        ds = ds.shuffle(len(idx), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH).map(lambda a, b: (tf.cast(a, tf.float32), tf.one_hot(b, len(CLASSES))),
                             num_parallel_calls=tf.data.AUTOTUNE)
    if train:
        ds = ds.map(lambda a, b: (augment(a, training=True), b), num_parallel_calls=tf.data.AUTOTUNE)
        if ALPHA > 0:
            ds = ds.map(mixup, num_parallel_calls=tf.data.AUTOTUNE)
        if CUTMIX_ALPHA > 0:
            ds = ds.map(cutmix, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.map(lambda a, b: (prep(a), b), num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

LOSS = keras.losses.CategoricalCrossentropy(label_smoothing=cfg['label_smoothing'])
base = ctor(include_top=False, weights='imagenet', input_shape=(IMG, IMG, 3)); base.trainable = False
inp = keras.Input((IMG, IMG, 3))
x = layers.GlobalAveragePooling2D()(base(inp, training=False))
model = keras.Model(inp, layers.Dense(len(CLASSES), activation='softmax', dtype='float32')(layers.Dropout(0.3)(x)))
# Training indices. With TARGET_PER_CLASS, every class is topped up to the target by repeating its
# TRAINING images; each repeat gets a different random augmentation every epoch, so it acts as an
# augmented copy. Validation and test images are never copied (no leakage).
rng = np.random.default_rng(SEED)
tr_idx = np.where(tr)[0]
if cfg.get('target_per_class'):
    parts = []
    for c in range(len(CLASSES)):
        own = np.where(tr & (y == c))[0]
        extra = rng.choice(own, max(cfg['target_per_class'] - len(own), 0), replace=True)
        parts += [own, extra]
    tr_idx = np.concatenate(parts)
ds_tr, ds_va, ds_te = make_ds(tr_idx, True), make_ds(np.where(va)[0], False), make_ds(np.where(te)[0], False)
cw = (dict(enumerate(compute_class_weight('balanced', classes=np.arange(len(CLASSES)), y=y[tr_idx])))
      if cfg.get('class_weights', True) else None)
opt = lambda lr: keras.optimizers.AdamW(learning_rate=lr, weight_decay=1e-4, clipnorm=1.0)

t0 = time.perf_counter()
model.compile(opt(cfg['head_lr']), LOSS, metrics=['accuracy'])
h1 = model.fit(ds_tr, validation_data=ds_va, epochs=cfg['head_epochs'], class_weight=cw, verbose=0,
               callbacks=[keras.callbacks.EarlyStopping('val_loss', patience=3, restore_best_weights=True)])
base.trainable = True
steps = int(np.ceil(len(tr_idx) / BATCH))
sched = keras.optimizers.schedules.CosineDecay(0.0, (cfg['ft_epochs'] - cfg['warmup_epochs']) * steps,
                                               warmup_target=cfg['ft_lr'], warmup_steps=cfg['warmup_epochs'] * steps)
model.compile(opt(sched), LOSS, metrics=['accuracy'])
h2 = model.fit(ds_tr, validation_data=ds_va, epochs=cfg['ft_epochs'], class_weight=cw, verbose=0,
               callbacks=[keras.callbacks.EarlyStopping('val_accuracy', mode='max', patience=cfg['ft_patience'],
                                                        restore_best_weights=True),
                          keras.callbacks.TerminateOnNaN()])
minutes = (time.perf_counter() - t0) / 60
pred = model.predict(ds_te, verbose=0).argmax(1); true = y[te]
vpred = model.predict(ds_va, verbose=0).argmax(1); vtrue = y[va]
LAB = range(len(CLASSES))
per = lambda fn, t, p: json.dumps(dict(zip(CLASSES, np.round(fn(t, p, average=None, labels=LAB, zero_division=0), 3).tolist())))

(OUT / 'history').mkdir(exist_ok=True); (OUT / 'preds').mkdir(exist_ok=True)
pd.concat([pd.DataFrame(h1.history).assign(phase='head'), pd.DataFrame(h2.history).assign(phase='finetune')]
          ).to_csv(OUT / 'history' / f'{name}.csv', index=False)
pd.DataFrame({'rel_path': d.rel_path[te].values, 'y_true': true, 'y_pred': pred}).to_csv(OUT / 'preds' / f'{name}.csv', index=False)
best = int(np.nanargmax(h2.history['val_accuracy']))
row = {'run': cfg['run'], 'crop': cfg['crop'], 'model': name, 'fold': FOLD, 'n_train': int(tr.sum()), 'n_train_effective': int(len(tr_idx)), 'augment': cfg.get('augment', 'basic'), 'mixup_alpha': ALPHA, 'cutmix_alpha': CUTMIX_ALPHA, 'n_val': int(va.sum()), 'n_test': int(te.sum()),
       'img_size': IMG, 'label_smoothing': cfg['label_smoothing'],
       'val_accuracy': accuracy_score(vtrue, vpred), 'val_macro_f1': f1_score(vtrue, vpred, average='macro', zero_division=0),
       'accuracy': accuracy_score(true, pred), 'balanced_accuracy': balanced_accuracy_score(true, pred),
       'macro_precision': precision_score(true, pred, average='macro', zero_division=0),
       'macro_recall': recall_score(true, pred, average='macro', zero_division=0),
       'macro_f1': f1_score(true, pred, average='macro', zero_division=0),
       'weighted_f1': f1_score(true, pred, average='weighted', zero_division=0),
       'cohen_kappa': cohen_kappa_score(true, pred), 'mcc': matthews_corrcoef(true, pred),
       'best_val_acc': float(h2.history['val_accuracy'][best]), 'train_acc_at_best': float(h2.history['accuracy'][best]),
       'epochs_head': len(h1.history['loss']), 'epochs_finetune': len(h2.history['loss']),
       'params_M': model.count_params() / 1e6, 'train_minutes': minutes,
       'per_class_precision': per(precision_score, true, pred), 'per_class_recall': per(recall_score, true, pred),
       'per_class_f1': per(f1_score, true, pred), 'val_per_class_f1': per(f1_score, vtrue, vpred),
       'confusion_matrix': json.dumps(confusion_matrix(true, pred, labels=range(len(CLASSES))).tolist())}
res = OUT / 'results.csv'
new = pd.DataFrame([row])
if res.exists():
    new = pd.concat([pd.read_csv(res), new], ignore_index=True)
new.to_csv(res, index=False)
print(f"{name:17s} VAL macroF1 {row['val_macro_f1']:.3f} | test acc {row['accuracy']:.3f} macroF1 {row['macro_f1']:.3f} | "
      f"train_acc {row['train_acc_at_best']:.3f} ({minutes:.1f} min)")

Writing /content/train_one.py


## 8. Train — one separate process per model

In [8]:
import subprocess
done = set(pd.read_csv(RESULTS_CSV).model) if RESULTS_CSV.exists() else set()
todo = [m for m in MODEL_NAMES if m not in done]
print(f'{len(done)} finished, {len(todo)} to do')
for i, name in enumerate(todo, 1):
    p = subprocess.run(['python', '/content/train_one.py', name, str(OUT)], capture_output=True, text=True)
    last = [l for l in p.stdout.strip().splitlines() if l.strip()]
    if p.returncode == 0 and last:
        print(f'[{i}/{len(todo)}] {last[-1]}   | RAM now {psutil.virtual_memory().used/1e9:.1f} GB')
    else:
        print(f'[{i}/{len(todo)}] {name} FAILED:\n' + p.stderr[-1500:])
print('Done.')

0 finished, 6 to do
[1/6] VGG16             VAL macroF1 0.691 | test acc 0.784 macroF1 0.739 | train_acc 0.713 (25.0 min)   | RAM now 1.6 GB
[2/6] ConvNeXtTiny      VAL macroF1 0.707 | test acc 0.757 macroF1 0.682 | train_acc 0.741 (23.7 min)   | RAM now 1.6 GB
[3/6] EfficientNetV2B0  VAL macroF1 0.707 | test acc 0.750 macroF1 0.722 | train_acc 0.569 (36.0 min)   | RAM now 2.0 GB
[4/6] DenseNet121       VAL macroF1 0.674 | test acc 0.743 macroF1 0.679 | train_acc 0.579 (36.2 min)   | RAM now 2.0 GB
[5/6] InceptionV3       VAL macroF1 0.669 | test acc 0.743 macroF1 0.687 | train_acc 0.661 (31.9 min)   | RAM now 2.0 GB
[6/6] ResNet50          VAL macroF1 0.690 | test acc 0.750 macroF1 0.694 | train_acc 0.690 (28.8 min)   | RAM now 2.0 GB
Done.


## 9. Results: validation (for selection) and test (development, reported)

In [9]:
r = pd.read_csv(RESULTS_CSV).drop_duplicates('model', keep='last').sort_values('val_macro_f1', ascending=False)
pct = lambda df, c: (100 * df[c]).round(2)
val = pd.DataFrame({'model': r.model, 'VAL macro-F1': pct(r, 'val_macro_f1'), 'VAL accuracy': pct(r, 'val_accuracy'),
                    'train acc at best': pct(r, 'train_acc_at_best'), 'epochs fine-tune': r.epochs_finetune})
test = pd.DataFrame({'model': r.model, 'accuracy': pct(r, 'accuracy'), 'macro precision': pct(r, 'macro_precision'),
                     'macro recall': pct(r, 'macro_recall'), 'macro-F1': pct(r, 'macro_f1'),
                     'balanced acc.': pct(r, 'balanced_accuracy'), 'MCC': pct(r, 'mcc')})
print(f'{RUN_NAME} | VALIDATION fold ({int(r.n_val.iloc[0])} images): used to compare models and settings'); display(val)
print(f'\n{RUN_NAME} | TEST fold 0 ({int(r.n_test.iloc[0])} images): development results, reported only'); display(test)

pc = {k: pd.DataFrame({m: json.loads(s) for m, s in zip(r.model, r[col])}).T * 100
      for k, col in [('precision', 'per_class_precision'), ('recall', 'per_class_recall'), ('F1', 'per_class_f1')]}
for k, t in pc.items():
    print(f'\nTest per-class {k} (%):'); display(t.round(1))

with pd.ExcelWriter(OUT / f'{RUN_NAME}_results.xlsx') as xw:
    val.to_excel(xw, sheet_name='validation', index=False)
    test.to_excel(xw, sheet_name='test_fold0', index=False)
    for k, t in pc.items():
        t.round(1).to_excel(xw, sheet_name=f'per_class_{k}')
    r.drop(columns=['confusion_matrix']).to_excel(xw, sheet_name='all_columns', index=False)
    pd.Series(cfg).astype(str).to_frame('value').to_excel(xw, sheet_name='protocol')
print('Saved', OUT / f'{RUN_NAME}_results.xlsx')

run07_cutmix | VALIDATION fold (154 images): used to compare models and settings


,model,VAL macro-F1,VAL accuracy,train acc at best,epochs fine-tune
1,ConvNeXtTiny,70.75,77.92,74.07,24
2,EfficientNetV2B0,70.67,75.32,56.93,37
0,VGG16,69.12,77.27,71.27,28
5,ResNet50,68.98,73.38,69.00,30
3,DenseNet121,67.38,72.73,57.93,19
4,InceptionV3,66.93,73.38,66.13,26



run07_cutmix | TEST fold 0 (148 images): development results, reported only


,model,accuracy,macro precision,macro recall,macro-F1,balanced acc.,MCC
1,ConvNeXtTiny,75.68,68.75,74.45,68.21,74.45,68.53
2,EfficientNetV2B0,75.00,71.53,74.21,72.21,74.21,66.39
0,VGG16,78.38,72.36,80.27,73.94,80.27,71.58
5,ResNet50,75.00,67.45,74.42,69.44,74.42,66.93
3,DenseNet121,74.32,66.70,72.38,67.86,72.38,65.91
4,InceptionV3,74.32,67.50,72.49,68.69,72.49,65.65



Test per-class precision (%):


,Tomato_Downy_Mildew,Tomato_Healthy,Tomato_Mosaic,Tomato_Spot,Tomato_White_Spot
ConvNeXtTiny,35.0,90.0,80.0,80.4,58.3
EfficientNetV2B0,77.8,81.8,73.1,75.0,50.0
VGG16,44.4,91.5,75.9,83.3,66.7
ResNet50,53.8,89.8,72.4,80.0,41.2
DenseNet121,50.0,86.3,75.0,76.1,46.2
InceptionV3,50.0,83.6,72.4,76.9,54.5



Test per-class recall (%):


,Tomato_Downy_Mildew,Tomato_Healthy,Tomato_Mosaic,Tomato_Spot,Tomato_White_Spot
ConvNeXtTiny,87.5,91.8,43.2,86.0,63.6
EfficientNetV2B0,87.5,91.8,51.4,76.7,63.6
VGG16,100.0,87.8,59.5,81.4,72.7
ResNet50,87.5,89.8,56.8,74.4,63.6
DenseNet121,87.5,89.8,48.6,81.4,54.5
InceptionV3,87.5,93.9,56.8,69.8,54.5



Test per-class F1 (%):


,Tomato_Downy_Mildew,Tomato_Healthy,Tomato_Mosaic,Tomato_Spot,Tomato_White_Spot
ConvNeXtTiny,50.0,90.9,56.1,83.1,60.9
EfficientNetV2B0,82.4,86.5,60.3,75.9,56.0
VGG16,61.5,89.6,66.7,82.4,69.6
ResNet50,66.7,89.8,63.6,77.1,50.0
DenseNet121,63.6,88.0,59.0,78.7,50.0
InceptionV3,63.6,88.5,63.6,73.2,54.5


Saved /content/drive/MyDrive/thesis/experiments/tomato/run07_cutmix/run07_cutmix_results.xlsx


## 10. Compare all runs that recorded validation macro-F1

In [10]:
runs = []
for f in sorted((PROJECT / 'experiments' / 'tomato').glob('*/results.csv')):
    x = pd.read_csv(f)
    if 'val_macro_f1' in x:
        runs.append(x.drop_duplicates('model', keep='last').assign(run=f.parent.name))
if runs:
    allr = pd.concat(runs)
    piv = (100 * allr.pivot_table(index='model', columns='run', values='val_macro_f1')).round(1)
    piv.loc['MEAN (models in every run)'] = piv.dropna().mean().round(1)   # fair: same models in each column
    print('VALIDATION macro-F1 (%) per model and run:'); display(piv)
    piv.to_csv(PROJECT / 'experiments' / 'tomato' / 'compare_validation_macro_f1.csv')

VALIDATION macro-F1 (%) per model and run:


run,run01b_baseline_top5,run03_labelsmooth,run04_balanced300,run05_balanced300_moderateaug,run06_mixup,run07_cutmix
model,,,,,,
ConvNeXtTiny,73.8,72.3,75.8,72.8,70.4,70.7
DenseNet121,71.0,66.8,68.9,73.6,69.0,67.4
EfficientNetV2B0,64.3,61.7,68.9,65.7,66.2,70.7
InceptionV3,64.1,63.8,66.0,69.9,68.3,66.9
ResNet50,68.7,NaN,69.7,75.0,68.2,69.0
VGG16,72.0,69.1,71.4,73.8,75.1,69.1
MEAN (models in every run),69.0,66.7,70.2,71.2,69.8,69.0
